# Feature Engineering & Transformation (Silver → Silver)

Inputs come from the cleaning & validation layer (`data/silver/*.parquet`).

This layer:

1. **ADI** — merges the three component metrics (claimant, crime, health) at `(lsoa_code, year)` and computes `ADI = claimant_rate + total_crime_rate + total_prevalence_rate`.
2. **House prices** — joins transactions to LSOA via the postcode lookup, derives `year` and `month`, and aggregates to `(lsoa_code, year, month)` with `median_price` and `transaction_count`.

Outputs:

- `data/silver/adi_transformed.parquet`
- `data/silver/houseprices_transformed.parquet`

The annual-vs-monthly temporal mismatch flagged in the brief is **not resolved here** — ADI stays annual, house prices stay monthly. The downstream/gold layer broadcasts ADI across months when it joins them.

## 1. Imports & Configuration

In [ ]:
import os

import numpy as np
import pandas as pd

SILVER_DIR = "../data/silver"

# Inputs from cleaning_and_validation_layer
CLAIMANT_IN = f"{SILVER_DIR}/claimant_clean.parquet"
CRIME_IN    = f"{SILVER_DIR}/crime_clean.parquet"
HEALTH_IN   = f"{SILVER_DIR}/health_clean.parquet"
HP_IN       = f"{SILVER_DIR}/houseprices_clean.parquet"
PC_IN       = f"{SILVER_DIR}/postcode_clean.parquet"

# Outputs
ADI_OUT = f"{SILVER_DIR}/adi_transformed.parquet"
HP_OUT  = f"{SILVER_DIR}/houseprices_transformed.parquet"

pd.set_option("display.max_columns", 50)

## 2. Reusable Utilities

In [ ]:
def add_date_parts(df, date_col):
    """Add `year` (int16) and `month` (int8) columns derived from `date_col`."""
    df = df.copy()
    dt = pd.to_datetime(df[date_col], errors="raise")
    df["year"]  = dt.dt.year.astype("int16")
    df["month"] = dt.dt.month.astype("int8")
    return df


def check_duplicates(df, keys, name):
    """Raise if `df` has duplicates on `keys`."""
    dup = df.duplicated(subset=keys).sum()
    if dup:
        raise AssertionError(f"[{name}] {dup:,} duplicate rows on {keys}")
    print(f"[{name}] no duplicates on {keys}  (rows={len(df):,})")


def check_row_counts(label, before, after):
    print(f"[{label}] rows {before:,} -> {after:,}  (delta {after - before:+,})")


def report(df, name):
    print(f"--- {name} ---")
    print(f"shape : {df.shape}")
    print(f"dtypes:\n{df.dtypes}")
    nulls = df.isna().sum()
    print(f"nulls :\n{nulls[nulls > 0] if nulls.sum() else 'none'}")
    print("head:")
    print(df.head(3))


def write_parquet(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"Written: {path}  shape={df.shape}")

---
## 3. ADI Transformation

Merge the three component metrics at `(lsoa_code, year)` and derive ADI.

`ADI = claimant_rate + total_crime_rate + total_prevalence_rate`

Inner join is intentional: ADI is only defined where all three components exist.

### 3a. Load component parquets

In [ ]:
claimant = pd.read_parquet(CLAIMANT_IN)
crime    = pd.read_parquet(CRIME_IN)
health   = pd.read_parquet(HEALTH_IN)

check_duplicates(claimant, ["lsoa_code", "year"], "claimant_clean")
check_duplicates(crime,    ["lsoa_code", "year"], "crime_clean")
check_duplicates(health,   ["lsoa_code", "year"], "health_clean")

### 3b. Merge components and compute ADI

In [ ]:
claimant_baseline = len(claimant)

adi = (
    claimant
    .merge(crime,  on=["lsoa_code", "year"], how="inner")
    .merge(health, on=["lsoa_code", "year"], how="inner")
)

check_row_counts("ADI inner-merge vs claimant baseline", claimant_baseline, len(adi))

adi["ADI"] = (
    adi["claimant_rate"]
    + adi["total_crime_rate"]
    + adi["total_prevalence_rate"]
).round(2)

# Keep only the columns we expose downstream; drop claimant_count (not part of ADI)
adi = adi[[
    "lsoa_code", "lsoa_name", "pop", "year",
    "claimant_rate", "total_crime_rate", "total_prevalence_rate", "ADI",
]]

### 3c. Validate and write

In [ ]:
check_duplicates(adi, ["lsoa_code", "year"], "ADI transformed")
report(adi, "ADI transformed")

write_parquet(adi, ADI_OUT)

---
## 4. House Prices Transformation

1. Load cleaned transactions and the 1:1 postcode→LSOA lookup.
2. Left-join to attach `lsoa_code`; assert no fan-out.
3. Derive `year` and `month` from `date_of_transfer`.
4. Drop rows with no LSOA match (cannot be assigned to a group), then drop `postcode` and aggregate to `(lsoa_code, year, month)` with `median_price` and `transaction_count`.

### 4a. Load and standardise dtypes

In [ ]:
hp        = pd.read_parquet(HP_IN)
postcodes = pd.read_parquet(PC_IN)

hp["date_of_transfer"] = pd.to_datetime(hp["date_of_transfer"], errors="raise")
hp["postcode"]         = hp["postcode"].astype("string")
hp["price"]            = hp["price"].astype("int64")

postcodes["postcode"]  = postcodes["postcode"].astype("string")
postcodes["lsoa_code"] = postcodes["lsoa_code"].astype("string")

report(hp,        "houseprices_clean")
report(postcodes, "postcode_clean")

### 4b. Join HP ← postcode lookup (no fan-out)

In [ ]:
check_duplicates(postcodes, ["postcode"], "postcode_clean")

before_join = len(hp)
hp = hp.merge(postcodes, on="postcode", how="left")
check_row_counts("HP left-join postcode_clean", before_join, len(hp))
assert len(hp) == before_join, "Fan-out detected on postcode join"

unmatched = hp["lsoa_code"].isna().sum()
print(f"Unmatched postcodes: {unmatched:,} ({unmatched / len(hp):.2%})")

### 4c. Derive time attributes and drop unassignable rows

In [ ]:
hp = add_date_parts(hp, "date_of_transfer")

before_drop = len(hp)
hp = hp.dropna(subset=["lsoa_code"]).reset_index(drop=True)
check_row_counts("HP drop unmatched LSOA", before_drop, len(hp))

### 4d. Aggregate to (lsoa_code, year, month)

In [ ]:
hp_agg = (
    hp.groupby(["lsoa_code", "year", "month"], as_index=False)
      .agg(
          median_price=("price", "median"),
          transaction_count=("price", "size"),
      )
)

hp_agg["transaction_count"] = hp_agg["transaction_count"].astype("int64")

### 4e. Validate and write

In [ ]:
check_duplicates(hp_agg, ["lsoa_code", "year", "month"], "Houseprices transformed")
assert (hp_agg["transaction_count"] >= 1).all(), "transaction_count must be >= 1 after groupby.size()"
report(hp_agg, "Houseprices transformed")

write_parquet(hp_agg, HP_OUT)

---
## 5. Run Summary

In [ ]:
outputs = {
    "adi_transformed.parquet":         ADI_OUT,
    "houseprices_transformed.parquet": HP_OUT,
}

print("=== Silver layer outputs (featureengineering_transformation) ===")
for name, path in outputs.items():
    df_check = pd.read_parquet(path)
    size_mb  = os.path.getsize(path) / 1_048_576
    print(f"  {name}: shape={df_check.shape}  size={size_mb:.2f} MB  path={path}")